In [2]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Install
# ═══════════════════════════════════════════════════════════
!pip install librosa soundfile numpy pandas torch scikit-learn kagglehub tqdm -q
print("Done")

Done


In [3]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Device + Config
# ═══════════════════════════════════════════════════════════
import torch

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-3
MAX_LEN    = 200   # same as before
NUM_CLASSES = 7    # neutral, happy, sad, angry, fear, disgust, surprise

EMOTION_LABELS = ["neutral", "happy", "sad", "angry", "fear", "disgust", "surprise"]
print(f"Device: {DEVICE}")

Device: cpu


In [4]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Download Datasets
# ═══════════════════════════════════════════════════════════
import kagglehub

# RAVDESS
ravdess_path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
print("RAVDESS:", ravdess_path)

# CREMA-D
cremad_path = kagglehub.dataset_download("ejlok1/cremad")
print("CREMA-D:", cremad_path)

# MELD — Multimodal EmotionLines Dataset
meld_path = kagglehub.dataset_download("zaber666/meld-dataset")
print("MELD:", meld_path)

/Users/tejaramidi/anaconda3/envs/emotion_nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 429M/429M [02:39<00:00, 2.81MB/s] 

Extracting files...


RAVDESS: /Users/tejaramidi/.cache/kagglehub/datasets/uwrfkaggler/ravdess-emotional-speech-audio/versions/1


100%|██████████| 451M/451M [02:37<00:00, 3.00MB/s] 

Extracting files...


CREMA-D: /Users/tejaramidi/.cache/kagglehub/datasets/ejlok1/cremad/versions/1


  1%|          | 109M/11.0G [00:38<1:05:54, 2.96MB/s] 


KeyboardInterrupt: 

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Parse RAVDESS
# ═══════════════════════════════════════════════════════════
import os
import pandas as pd

# RAVDESS emotion codes → unified labels
RAVDESS_MAP = {
    1: "neutral",   # neutral
    2: "neutral",   # calm → merge into neutral
    3: "happy",
    4: "sad",
    5: "angry",
    6: "fear",
    7: "disgust",
    8: "surprise"
}

def parse_ravdess(path):
    data = []
    for actor in os.listdir(path):
        actor_path = os.path.join(path, actor)
        if not os.path.isdir(actor_path):
            continue
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                parts   = file.split("-")
                code    = int(parts[2])
                emotion = RAVDESS_MAP.get(code)
                if emotion:
                    data.append({
                        "path":    os.path.join(actor_path, file),
                        "emotion": emotion,
                        "source":  "ravdess"
                    })
    return pd.DataFrame(data)

ravdess_audio = os.path.join(ravdess_path, "audio_speech_actors_01-24")
df_ravdess    = parse_ravdess(ravdess_audio)
print(f"RAVDESS: {len(df_ravdess)} samples")
print(df_ravdess["emotion"].value_counts())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Parse CREMA-D
# ═══════════════════════════════════════════════════════════

# CREMA-D filename format: 1001_DFA_ANG_XX.wav
# 3rd segment = emotion code
CREMAD_MAP = {
    "ANG": "angry",
    "DIS": "disgust",
    "FEA": "fear",
    "HAP": "happy",
    "NEU": "neutral",
    "SAD": "sad"
    # no surprise in CREMA-D
}

def parse_cremad(path):
    data = []
    # find the AudioWAV folder
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith(".wav"):
                parts   = file.split("_")
                if len(parts) < 3:
                    continue
                code    = parts[2]
                emotion = CREMAD_MAP.get(code)
                if emotion:
                    data.append({
                        "path":    os.path.join(root, file),
                        "emotion": emotion,
                        "source":  "cremad"
                    })
    return pd.DataFrame(data)

df_cremad = parse_cremad(cremad_path)
print(f"CREMA-D: {len(df_cremad)} samples")
print(df_cremad["emotion"].value_counts())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Parse MELD (audio files only)
# ═══════════════════════════════════════════════════════════
import glob

# MELD emotion labels
MELD_MAP = {
    "neutral":   "neutral",
    "joy":       "happy",
    "sadness":   "sad",
    "anger":     "angry",
    "fear":      "fear",
    "disgust":   "disgust",
    "surprise":  "surprise"
}

def parse_meld(path):
    data = []
    # MELD has train/dev/test CSV files with emotion labels
    for split in ["train", "dev", "test"]:
        csv_path = os.path.join(path, f"{split}_sent_emo.csv")
        if not os.path.exists(csv_path):
            # try alternate names
            candidates = glob.glob(os.path.join(path, f"*{split}*.csv"))
            if not candidates:
                continue
            csv_path = candidates[0]

        df_csv = pd.read_csv(csv_path)

        # Look for audio files
        audio_dir = os.path.join(path, split)
        if not os.path.exists(audio_dir):
            continue

        for _, row in df_csv.iterrows():
            emotion = MELD_MAP.get(str(row.get("Emotion", "")).lower())
            if not emotion:
                continue

            # MELD audio files named by dialogue_id and utterance_id
            dia_id  = row.get("Dialogue_ID", "")
            utt_id  = row.get("Utterance_ID", "")
            audio_file = os.path.join(audio_dir, f"dia{dia_id}_utt{utt_id}.wav")

            if os.path.exists(audio_file):
                data.append({
                    "path":    audio_file,
                    "emotion": emotion,
                    "source":  "meld"
                })

    return pd.DataFrame(data)

df_meld = parse_meld(meld_path)
print(f"MELD: {len(df_meld)} samples")
if len(df_meld) > 0:
    print(df_meld["emotion"].value_counts())
else:
    print("MELD audio files not found — check dataset structure, will proceed without MELD")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Combine + Balance
# ═══════════════════════════════════════════════════════════
import numpy as np

# Combine all available datasets
dfs = [df_ravdess, df_cremad]
if len(df_meld) > 0:
    dfs.append(df_meld)

df = pd.concat(dfs, ignore_index=True)
df = df[df["emotion"].isin(EMOTION_LABELS)].reset_index(drop=True)

# Map emotion string to index
LABEL2IDX = {e: i for i, e in enumerate(EMOTION_LABELS)}
df["label"] = df["emotion"].map(LABEL2IDX)

print(f"\nTotal combined: {len(df)} samples")
print("\nEmotion distribution:")
print(df["emotion"].value_counts())

# Cap dominant classes to 3000 max to reduce imbalance
MAX_PER_CLASS = 3000
df_balanced = df.groupby("emotion").apply(
    lambda x: x.sample(min(len(x), MAX_PER_CLASS), random_state=42)
).reset_index(drop=True)

print(f"\nAfter balancing (max {MAX_PER_CLASS} per class):")
print(df_balanced["emotion"].value_counts())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Feature Extraction (MFCC + Delta + Delta2 + Pitch + ZCR + RMS)
# Better than before — adds tonal features for tone detection
# ═══════════════════════════════════════════════════════════
import librosa
import numpy as np

def extract_features(file_path, max_len=200):
    """
    Extracts rich acoustic features:
    - MFCC (40) — vocal tract shape
    - Delta MFCC (40) — rate of change
    - Delta-Delta MFCC (40) — acceleration
    - Pitch/F0 (1 row, repeated) — tone high/low
    - ZCR (1 row) — energy level
    - RMS Energy (1 row) — loudness
    Total: 123 feature rows × 200 frames
    """
    try:
        y, sr = librosa.load(file_path, sr=22050, duration=5.0)

        if len(y) == 0:
            return None

        # MFCC + derivatives — captures spectral + temporal patterns
        mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        delta  = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)

        # Pitch F0 — KEY for tone detection (sad=low, excited=high)
        f0, voiced_flag, _ = librosa.pyin(
            y, fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr
        )
        f0 = np.nan_to_num(f0)  # replace NaN with 0
        # Reshape to (1, frames) to stack with MFCC
        f0_row = f0[:mfcc.shape[1]] if len(f0) >= mfcc.shape[1] else np.pad(f0, (0, mfcc.shape[1] - len(f0)))
        f0_row = f0_row.reshape(1, -1)

        # ZCR — high energy states (anger, excitement)
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr = zcr[:, :mfcc.shape[1]] if zcr.shape[1] >= mfcc.shape[1] else np.pad(zcr, ((0,0),(0, mfcc.shape[1]-zcr.shape[1])))

        # RMS energy — loudness (sad=quiet, angry=loud)
        rms = librosa.feature.rms(y=y)
        rms = rms[:, :mfcc.shape[1]] if rms.shape[1] >= mfcc.shape[1] else np.pad(rms, ((0,0),(0, mfcc.shape[1]-rms.shape[1])))

        # Stack all features → (123, frames)
        features = np.vstack([mfcc, delta, delta2, f0_row, zcr, rms])

        # Normalize per sample
        features = (features - np.mean(features)) / (np.std(features) + 1e-6)

        # Pad or truncate to max_len
        if features.shape[1] < max_len:
            features = np.pad(features, ((0, 0), (0, max_len - features.shape[1])))
        else:
            features = features[:, :max_len]

        return features.astype(np.float32)

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Test feature extraction
sample = df_balanced.iloc[0]["path"]
feat   = extract_features(sample)
print(f"Feature shape: {feat.shape}")  # should be (123, 200)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Pre-extract all features (save time during training)
# ═══════════════════════════════════════════════════════════
from tqdm import tqdm

print("Extracting features for all samples...")
features_list = []
labels_list   = []
valid_idx     = []

for i, row in tqdm(df_balanced.iterrows(), total=len(df_balanced)):
    feat = extract_features(row["path"])
    if feat is not None:
        features_list.append(feat)
        labels_list.append(row["label"])
        valid_idx.append(i)

features_array = np.array(features_list)  # (N, 123, 200)
labels_array   = np.array(labels_list)

print(f"Valid samples: {len(features_array)}")
print(f"Features shape: {features_array.shape}")

# Save to disk — avoid re-extracting on resume
np.save("features.npy", features_array)
np.save("labels.npy",   labels_array)
print("Saved features.npy and labels.npy")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Dataset + DataLoader
# ═══════════════════════════════════════════════════════════
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Load pre-extracted features
features_array = np.load("features.npy")
labels_array   = np.load("labels.npy")

# 80 / 10 / 10 stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    features_array, labels_array, test_size=0.20,
    random_state=42, stratify=labels_array
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

class SpeechDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (N, 1, 123, 200)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SpeechDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SpeechDataset(X_val,   y_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(SpeechDataset(X_test,  y_test),  batch_size=BATCH_SIZE)

In [5]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — Model (2D-CNN, same architecture, bigger now)
# ═══════════════════════════════════════════════════════════
import torch.nn as nn

class SpeechEmotionModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.conv = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),
        )

        # Calculate FC input size
        # Input: (1, 123, 200) → after 3 MaxPool2d(2): (128, 15, 25)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 15 * 25, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.fc(self.conv(x))

model = SpeechEmotionModel(num_classes=NUM_CLASSES).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Verify output shape
dummy = torch.zeros(2, 1, 123, 200).to(DEVICE)
out   = model(dummy)
print(f"Output shape: {out.shape}")  # should be (2, 7)

Parameters: 12,398,279
Output shape: torch.Size([2, 7])


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — Loss + Optimizer + Scheduler
# ═══════════════════════════════════════════════════════════

# Class weights for remaining imbalance
class_counts = np.bincount(y_train, minlength=NUM_CLASSES)
class_weights = torch.tensor(
    len(y_train) / (NUM_CLASSES * class_counts),
    dtype=torch.float32
).to(DEVICE)

print("Class weights:", {EMOTION_LABELS[i]: round(float(class_weights[i]),2) for i in range(NUM_CLASSES)})

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# Reduce LR when val accuracy plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, verbose=True
)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — Training Loop with checkpoint + resume
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import f1_score
import json

CHECKPOINT_PATH = "speech_checkpoint.pth"
BEST_MODEL_PATH = "best_speech_model.pth"
PATIENCE        = 5

start_epoch    = 0
best_val_acc   = 0.0
patience_count = 0

# Resume if checkpoint exists
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch   = ckpt["epoch"] + 1
    best_val_acc  = ckpt["best_val_acc"]
    patience_count = ckpt["patience_count"]
    print(f"Resumed from epoch {start_epoch} | Best acc: {best_val_acc:.4f}")
else:
    print("Starting fresh training")

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y  = X.to(DEVICE), y.to(DEVICE)
            preds = model(X).argmax(dim=1)
            correct += (preds == y).sum().item()
            total   += y.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    acc     = correct / total
    f1_mac  = f1_score(all_labels, all_preds, average="macro",  zero_division=0)
    f1_mic  = f1_score(all_labels, all_preds, average="micro",  zero_division=0)
    return acc, f1_mac, f1_mic

history = []

for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0

    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss          = total_loss / len(train_loader)
    val_acc, f1_mac, f1_mic = evaluate(model, val_loader)
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | Macro-F1: {f1_mac:.4f}")

    history.append({
        "epoch": epoch+1, "loss": round(avg_loss,4),
        "val_acc": round(val_acc,4), "macro_f1": round(f1_mac,4)
    })

    # Save checkpoint every epoch
    torch.save({
        "epoch":              epoch,
        "model_state_dict":   model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_acc":       best_val_acc,
        "patience_count":     patience_count
    }, CHECKPOINT_PATH)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc   = val_acc
        patience_count = 0
        torch.save({"model_state_dict": model.state_dict()}, BEST_MODEL_PATH)
        print(f"  Best model saved! ({best_val_acc:.4f})")
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print("Early stopping.")
            break

print(f"\nTraining complete. Best Val Acc: {best_val_acc:.4f}")
pd.DataFrame(history)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14 — Test Set Evaluation
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import classification_report

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE)["model_state_dict"])
test_acc, test_f1_mac, test_f1_mic = evaluate(model, test_loader)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro-F1: {test_f1_mac:.4f}")
print(f"Test Micro-F1: {test_f1_mic:.4f}")

# Per-class report
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        all_preds.extend(model(X).argmax(dim=1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())

print("\nPer-class report:")
print(classification_report(all_labels, all_preds, target_names=EMOTION_LABELS, zero_division=0))

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 15 — Update speech_pipeline.py with mismatch detection
# ═══════════════════════════════════════════════════════════
# After retraining, update get_acoustic_emotion() in speech_pipeline.py:
# 1. New model has 123 feature rows not 120 — update extract_features() to match
# 2. New model has 7 classes not 8 (calm removed)
# 3. Add mismatch detection

# Mismatch detection logic to add in run_speech_pipeline():

POSITIVE_EMOTIONS = {"joy", "trust", "love", "optimism", "anticipation"}
NEGATIVE_EMOTIONS = {"sadness", "fear", "anger", "disgust", "pessimism"}

ACOUSTIC_TO_LINGUISTIC = {
    "sad":     "sadness",
    "angry":   "anger",
    "fear":    "fear",
    "disgust": "disgust",
    "happy":   "joy",
    "neutral": None,      # keep linguistic if acoustic is neutral
    "surprise":"surprise"
}

def detect_mismatch_and_resolve(dominant_linguistic, dominant_acoustic, acoustic_conf):
    """
    Detects verbal-vocal mismatch and resolves to final emotion.
    Example: words say 'joy' but voice tone says 'sad' → use 'sad'
    """
    ling_valence = "positive" if dominant_linguistic in POSITIVE_EMOTIONS else "negative"
    acou_valence = "positive" if dominant_acoustic == "happy" else (
                   "neutral"  if dominant_acoustic == "neutral" else "negative")

    mismatch = (ling_valence == "positive" and acou_valence == "negative") or \
               (ling_valence == "negative" and acou_valence == "positive")

    # Resolve final emotion
    if mismatch and acoustic_conf >= 0.35:
        # Tone wins over words when confidence is reasonable
        override = ACOUSTIC_TO_LINGUISTIC.get(dominant_acoustic)
        final    = override if override else dominant_linguistic
    else:
        final = dominant_linguistic

    return mismatch, final

print("Mismatch detection logic ready.")
print("Add detect_mismatch_and_resolve() to speech_pipeline.py after retraining.")